# Project: Image Style Transfer

## 1. Problem Statement

The goal of this project is to create a system that applies the artistic style of Vincent van Gogh to ordinary photographs. The model should preserve the main content and structure of the original image while transforming its visual appearance to match Van Gogh’s painting style. Deep learning techniques are used to learn and transfer characteristic patterns such as colors, brush strokes, and textures. The final system should generate visually appealing stylized images in an efficient and automatic way.

## 2. Import Libraries and Define Constants

In [1]:
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
import torch.optim as optim
import uuid
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms, models
from tqdm import tqdm
from PIL import Image

is_debug_mode = False
print('Mode:', 'debug' if is_debug_mode else 'release')

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

img_size = 256 if is_debug_mode else 640
print('Image size: ', img_size)

num_epochs = 10 if is_debug_mode else 10
print('Num epochs:', num_epochs)

batch_size = 4

Mode: release
Using device: mps
Image size:  640
Num epochs: 10


## 3. Define Encoder–Decoder CNN

This model is a feed-forward convolutional neural network that transforms a content image into a stylized version in a single pass. It encodes the image into a lower-resolution feature space, applies learned style transformations, and then upsamples it back to the original resolution. During training, it learns to preserve content while matching the style (e.g., Van Gogh) using perceptual losses from a pretrained VGG network.

In [2]:
class StyleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 9, padding=4),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),      # downsample
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),     # downsample
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            # Upsample back to original size
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(32, 3, 9, padding=4),
            nn.Tanh()  # output in [-1, 1]
        )

    def forward(self, x):
        return self.net(x)

This module wraps a pretrained VGG19 network to extract intermediate feature maps used for perceptual losses in style transfer. It forwards the input image through selected layers and returns activations from specific depths, which represent different levels of visual abstraction (edges, textures, structures). The network is frozen (no gradient updates) because it serves only as a fixed feature extractor for computing content and style losses.

In [ ]:
class VGGFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features
        self.layers = nn.ModuleList(vgg[:29])
        self.targets = [0, 5, 10, 19, 28]

        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        feats = []
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i in self.targets:
                feats.append(x)
        return feats

Computes the Gram matrix of a batch of feature maps, which captures the correlations between channels (i.e., texture/style information) independent of spatial layout. The feature map [B, C, H, W] is reshaped to [B, C, H*W], and a batch matrix multiplication produces a [B, C, C] matrix representing channel-wise similarities. The result is normalized by the number of elements (C * H * W) for scale stability.

In [ ]:
def gram_matrix(f):
    B, C, H, W = f.size()
    f = f.view(B, C, H * W)
    g = torch.bmm(f, f.transpose(1, 2))
    return g / (C * H * W)

## 4. Load Content and Style images

Create a content and a style dataset and data loaders and apply same transformations to them. Content images are a training base of imagese to learn applicaiton of style. Style images are a set of 268 pictures of Van Gogh.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(img_size+20),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

content_dataset = ImageFolder("data/content/", transform=transform)
content_loader = DataLoader(content_dataset, batch_size=4, shuffle=True)

style_dataset = ImageFolder("data/style", transform=transform)
style_loader = DataLoader(style_dataset, batch_size=1, shuffle=False)


## 5. Initialize the Models

Initialize the style and feature extraction models. Pick and initialize the optimizer and the loss function.

In [6]:
style_net = StyleNet().to(device)
vgg = VGGFeatures().to(device).eval()

optimizer = optim.Adam(style_net.parameters(), lr=1e-3)
mse = nn.MSELoss()

This code computes the average style representation from a set of style images by accumulating their Gram matrices. Each style image is passed through the VGG feature extractor, and Gram matrices are calculated to capture texture and color statistics at multiple layers. These matrices are summed across all images and then averaged, producing a single set of target style statistics used during training.

In [7]:
style_grams_sum = None
num_style_images = 0

with torch.no_grad():
    for style_img, _ in style_loader:
        style_img = style_img.to(device)

        style_feats = vgg(style_img)
        grams = [gram_matrix(f) for f in style_feats]

        if style_grams_sum is None:
            style_grams_sum = grams
        else:
            for i in range(len(grams)):
                style_grams_sum[i] += grams[i]

        num_style_images += 1

# Compute mean Gram
style_grams = [g / num_style_images for g in style_grams_sum]



## 6. Training Loop

This training loop optimizes the StyleNet to transform content images into stylized outputs by balancing content preservation and style transfer. For each batch, the model generates stylized images, and both the original and stylized images are passed through a frozen VGG network to extract feature representations. Content loss measures how well the structure is preserved, while style loss compares Gram matrices to match the target style statistics, and their weighted combination is used to update the model parameters via backpropagation.

In [8]:
STYLE_WEIGHT = 1e5
CONTENT_WEIGHT = 1.0

for epoch in range(num_epochs):
    for content_imgs, _ in content_loader:
        content_imgs = content_imgs.to(device)

        stylized = style_net(content_imgs)

        content_feats = vgg(content_imgs)
        stylized_feats = vgg(stylized)

        # Content loss (relu3_1)
        content_loss = mse(
            stylized_feats[2],
            content_feats[2]
        )

        # Style loss
        style_loss = 0.0
        for sf, sg in zip(stylized_feats, style_grams):
            G = gram_matrix(sf)
            sg_expanded = sg.expand_as(G)
            style_loss += mse(G, sg_expanded)
            # print("Style diff:", torch.mean(torch.abs(G - sg_expanded)).item())


        loss = CONTENT_WEIGHT * content_loss + STYLE_WEIGHT * style_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} | Content {content_loss.item():.5f} | Style {style_loss.item():.5f} | Style weighted {STYLE_WEIGHT*style_loss.item():.5f}")


Epoch 1 | Content 3.10207 | Style 0.00035 | Style weighted 35.21910
Epoch 2 | Content 11.00275 | Style 0.00032 | Style weighted 32.06508
Epoch 3 | Content 12.03553 | Style 0.00022 | Style weighted 22.10078
Epoch 4 | Content 5.82712 | Style 0.00024 | Style weighted 24.12040
Epoch 5 | Content 6.04736 | Style 0.00019 | Style weighted 18.56825
Epoch 6 | Content 5.45363 | Style 0.00017 | Style weighted 16.74136
Epoch 7 | Content 11.35945 | Style 0.00019 | Style weighted 19.22971
Epoch 8 | Content 7.60659 | Style 0.00012 | Style weighted 12.21864
Epoch 9 | Content 11.60811 | Style 0.00008 | Style weighted 8.40477
Epoch 10 | Content 11.21650 | Style 0.00009 | Style weighted 8.74435


Save the model for any later usage, e.g. from a different application.

In [ ]:
torch.save(style_net.state_dict(), "vangogh_style_net.pth")

In [ ]:
def apply_style(model_path, image_path, result_image_path):
    model = StyleNet().to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()

    img = Image.open(image_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(img)

    def denorm(x):
        mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(device)
        std  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(device)
        return (x * std + mean).clamp(0,1)

    out_img = denorm(out)[0].cpu()
    transforms.ToPILImage()(out_img).save(result_image_path)


# apply_style("vangogh_style_net.pth", "data/london.jpg", "data/london_result"+uuid.uuid4().hex[:6]+".jpg")
# apply_style("vangogh_style_net.pth", "data/munich.jpg", "data/munich_result"+uuid.uuid4().hex[:6]+".jpg")
# apply_style("vangogh_style_net.pth", "data/paris.jpg", "data/paris_result"+uuid.uuid4().hex[:6]+".jpg")
# apply_style("vangogh_style_net.pth", "data/tokyo.jpg", "data/tokyo_result"+uuid.uuid4().hex[:6]+".jpg")
# apply_style("vangogh_style_net.pth", "data/cats.jpg", "data/cats_result"+uuid.uuid4().hex[:6]+".jpg")
# apply_style("vangogh_style_net.pth", "data/dog.jpg", "data/dog_result"+uuid.uuid4().hex[:6]+".jpg")